In [99]:

from pyspark.sql import SparkSession
'''
We have
 -pyspark.sql
 -pyspark.RDD
 -pyspark.ml
 -pyspark.streaming
 -pyspark.pandas

'''
spark =SparkSession.builder.appName("FirstSparkApp").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [100]:
data=[
    ('Alice', 100),
    ('Bob', 200),
    ('Charlie', 300),
]
column=["Name","Score"]

df=spark.createDataFrame(data,column)

df.show()

+-------+-----+
|   Name|Score|
+-------+-----+
|  Alice|  100|
|    Bob|  200|
|Charlie|  300|
+-------+-----+



In [101]:
from pyspark.sql import DataFrame

df:DataFrame=(spark.read
    .option("inferSchema","true")
    .option("header","true")
    .csv("../data_sample/customers-10000.csv"))


## Check number of columns

In [102]:
df.columns

['Index',
 'Customer Id',
 'First Name',
 'Last Name',
 'Company',
 'City',
 'Country',
 'Phone 1',
 'Phone 2',
 'Email',
 'Subscription Date',
 'Website']

## Column rename

In [103]:
for column in df.columns:
    df=df.withColumnRenamed(column,column.replace(" ","_").lower())

df.columns

['index',
 'customer_id',
 'first_name',
 'last_name',
 'company',
 'city',
 'country',
 'phone_1',
 'phone_2',
 'email',
 'subscription_date',
 'website']

## sql DataFrame select example

In [104]:
df.select("first_name","last_name").show()

+----------+---------+
|first_name|last_name|
+----------+---------+
|   Heather| Callahan|
|  Kristina|  Ferrell|
|    Briana| Andersen|
|     Patty|    Ponce|
|  Kathleen|Mccormick|
|    Trevor|      Lee|
|    Mathew|  Hoffman|
|     Glenn|  Wiggins|
|     Bruce|    Payne|
|   Brendan|   Franco|
|    Martin|  Hawkins|
|      Sara|  Shaffer|
|      Dave|    Moran|
|      Glen|  Hammond|
| Catherine|Blackwell|
|     Larry|   Newton|
|     Danny|   Archer|
|       Kim|  Griffin|
|   Kristin| Valencia|
|    Hannah|   Ramsey|
+----------+---------+
only showing top 20 rows


## SQL code example on sql DataFrame

In [105]:

df.createOrReplaceTempView("customers")
result =spark.sql("select first_name, company, city from customers ")
result.show()

+----------+--------------------+------------------+
|first_name|             company|              city|
+----------+--------------------+------------------+
|   Heather|        Mosley-David|  Lake Jeffborough|
|  Kristina|Horn, Shepard and...|        Aaronville|
|    Briana|         Irwin-Oneal|       East Jordan|
|     Patty|    Richardson Group|  East Kristintown|
|  Kathleen|        Carson-Burch|       Andresmouth|
|    Trevor|        Maddox Group|Lake Madelineburgh|
|    Mathew|Bender, Pittman a...|        West Ralph|
|     Glenn|        Glenn-Harvey|        Ambershire|
|     Bruce|Arroyo, Cain and ...|       Barrettview|
|   Brendan|Schaefer, Blair a...|        New Rickey|
|    Martin|           Lopez Inc|       Lake Bobton|
|      Sara|Dudley, Coleman a...|           Orrland|
|      Dave|     Harrell-Donovan|   South Elizabeth|
|      Glen|Schaefer, Chung a...|        Pamelatown|
| Catherine|Mack, Garcia and ...|         Lake Seth|
|     Larry|           Downs PLC|          Man

In [106]:
print(f' total records count {df.count()}')
# Explain plan
df.explain()

 total records count 10000
== Physical Plan ==
*(1) Project [Index#22027 AS index#22040, Customer Id#22028 AS customer_id#22041, First Name#22029 AS first_name#22042, Last Name#22030 AS last_name#22043, Company#22031 AS company#22044, City#22032 AS city#22045, Country#22033 AS country#22046, Phone 1#22034 AS phone_1#22047, Phone 2#22035 AS phone_2#22048, Email#22036 AS email#22049, Subscription Date#22037 AS subscription_date#22050, Website#22038 AS website#22051]
+- FileScan csv [Index#22027,Customer Id#22028,First Name#22029,Last Name#22030,Company#22031,City#22032,Country#22033,Phone 1#22034,Phone 2#22035,Email#22036,Subscription Date#22037,Website#22038] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/feroz/Desktop/python/first_ml/data_sample/customers-10000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Index:int,Customer Id:string,First Name:string,Last Name:string,Company:string,City:string...




## Filter example

In [107]:
ndf =df.filter(df.country =='Norway')
ndf.select('first_name','last_name','country').show()

+----------+---------+-------+
|first_name|last_name|country|
+----------+---------+-------+
|   Heather| Callahan| Norway|
|  Danielle|    Huang| Norway|
|     Danny|     Hall| Norway|
|     Maria|   Conrad| Norway|
|    Kristi|   Cortez| Norway|
|      Jack|  Robbins| Norway|
| Christian| Thornton| Norway|
|    Andres|     Nash| Norway|
|     Traci|   Cooper| Norway|
|     Cathy|   Jarvis| Norway|
|    Denise|   Medina| Norway|
|   Patrick|      Ray| Norway|
|    Morgan|   Robles| Norway|
|     Randy|    Rivas| Norway|
|     Darin|     Frye| Norway|
|     Donna|    Tyler| Norway|
|     Jerry|   Howard| Norway|
|   Suzanne|  Vasquez| Norway|
|    Melvin|  Rodgers| Norway|
|  Stefanie| Castillo| Norway|
+----------+---------+-------+
only showing top 20 rows


## Concat , New column extraction , withColumn

In [108]:
from pyspark.sql.functions import  col, concat, lit, when
df= df.withColumn('full_name',concat(col('first_name'),lit(" "),col('last_name')))
df.select("full_name","first_name","last_name").show()

+-------------------+----------+---------+
|          full_name|first_name|last_name|
+-------------------+----------+---------+
|   Heather Callahan|   Heather| Callahan|
|   Kristina Ferrell|  Kristina|  Ferrell|
|    Briana Andersen|    Briana| Andersen|
|        Patty Ponce|     Patty|    Ponce|
| Kathleen Mccormick|  Kathleen|Mccormick|
|         Trevor Lee|    Trevor|      Lee|
|     Mathew Hoffman|    Mathew|  Hoffman|
|      Glenn Wiggins|     Glenn|  Wiggins|
|        Bruce Payne|     Bruce|    Payne|
|     Brendan Franco|   Brendan|   Franco|
|     Martin Hawkins|    Martin|  Hawkins|
|       Sara Shaffer|      Sara|  Shaffer|
|         Dave Moran|      Dave|    Moran|
|       Glen Hammond|      Glen|  Hammond|
|Catherine Blackwell| Catherine|Blackwell|
|       Larry Newton|     Larry|   Newton|
|       Danny Archer|     Danny|   Archer|
|        Kim Griffin|       Kim|  Griffin|
|   Kristin Valencia|   Kristin| Valencia|
|      Hannah Ramsey|    Hannah|   Ramsey|
+----------

In [109]:
print('existing columns are',df.columns)

df=df.drop('phone_2')
print('after dropping columns phone 2',df.columns)


existing columns are ['index', 'customer_id', 'first_name', 'last_name', 'company', 'city', 'country', 'phone_1', 'phone_2', 'email', 'subscription_date', 'website', 'full_name']
after dropping columns phone 2 ['index', 'customer_id', 'first_name', 'last_name', 'company', 'city', 'country', 'phone_1', 'email', 'subscription_date', 'website', 'full_name']


## Rename phone_1 to phone

In [110]:
print(df.columns)
df=df.withColumnRenamed('phone_1','phone')
print(df.columns)

['index', 'customer_id', 'first_name', 'last_name', 'company', 'city', 'country', 'phone_1', 'email', 'subscription_date', 'website', 'full_name']
['index', 'customer_id', 'first_name', 'last_name', 'company', 'city', 'country', 'phone', 'email', 'subscription_date', 'website', 'full_name']


# Data Exploration

# topic
- Describe
- missing value [df.na.drop(), df.na.fill()]
- Unique values
- counting category

In [115]:
# Describe
df.describe().show()
# Map
df =(df
     .withColumn("phone",
                 when(col('phone')=='986-340-0253', lit(None))
                 .otherwise(col('phone'))
                 )
     )
# Single null count by column
df.filter(col('phone').isNull()).show()
print("total null counts")
null_columns =[ {cols:df.filter(col(cols).isNull()).count() } for cols in df.columns ]
print(null_columns)

# Unique value counts
# For a single columns
print("\n\nUnique count ")
print("Country unique",df.select('country').distinct().count())
# for many columns

col_distinct_count =[{cols : df.select(cols).distinct().count()} for cols in df.columns ]
print("All columns unique value count")
print(col_distinct_count)

+-------+------------------+---------------+----------+---------+------------+-----------+-----------+--------------------+--------------------+--------------------+----------+
|summary|             index|    customer_id|first_name|last_name|     company|       city|    country|               phone|               email|             website| full_name|
+-------+------------------+---------------+----------+---------+------------+-----------+-----------+--------------------+--------------------+--------------------+----------+
|  count|             10000|          10000|     10000|    10000|       10000|      10000|      10000|                9999|               10000|               10000|     10000|
|   mean|            5000.5|       Infinity|      NULL|     NULL|        NULL|       NULL|       NULL| 5.012894285047503E9|                NULL|                NULL|      NULL|
| stddev|2886.8956799071675|           NULL|      NULL|     NULL|        NULL|       NULL|       NULL|2.86949477761